# PureScale: Classical Image Enhancer (Pure Computer Vision)
- Zero AI / No Heavy Models: Runs in milliseconds using pure OpenCV & NumPy on CPU.
- No GPU Needed: Uses 0% of your GPU quota.
- Fully Controllable: Detailed sliders for sharpness, denoise intensity, adaptive contrast, brightness, warmth, and vibrance.

In [ ]:
# @title PureScale: Image Quality Enhancer Settings

# @markdown ---
# @markdown ### [1] Resolution & Size
# @markdown Multiplies the width and height of the image (default: 3).
# @markdown - **1**: Keeps original size (enhances quality only, no size change)
# @markdown - **2**: 2x larger (doubles width and height)
# @markdown - **3**: 3x larger (e.g., 1000x1000px becomes 3000x3000px)
# @markdown - **4**: 4x larger (great for small icons or low-res graphics)
upscale_factor = 3 # @param [1, 1.5, 2, 2.5, 3, 3.5, 4, 6, 8] {type:"raw"}

# @markdown ---
# @markdown ### [2] Edge Sharpness & Detail Radius
# @markdown **sharpen_strength** (default: 1.2): Makes soft or blurry edges crisp.
# @markdown - **0.0**: No sharpening
# @markdown - **0.8 - 1.4**: Natural, crisp clarity (recommended)
# @markdown - **2.0+**: Aggressive sharpness (may show white edge halos)
sharpen_strength = 1.2 # @param {"type":"slider","min":0.0,"max":3.0,"step":0.1}

# @markdown **sharpen_radius** (default: 2.5): Controls which details get sharpened.
# @markdown - **1.0 - 2.0**: Fine details (eyelashes, hair strands, fabric weave, small text)
# @markdown - **3.0 - 5.0**: Broad outlines (larger objects, silhouettes, structural edges)
sharpen_radius = 2.5 # @param {"type":"slider","min":1.0,"max":6.0,"step":0.5}

# @markdown ---
# @markdown ### [3] Noise & Grain Cleaning
# @markdown **enable_denoise** (default: True): Turn bilateral noise reduction on or off.
enable_denoise = True # @param {type:"boolean"}

# @markdown **denoise_intensity** (default: 50): How aggressively to smooth out grain.
# @markdown - **20 - 40**: Light cleanup (preserves fine natural film grain)
# @markdown - **50 - 60**: Balanced (cleans phone camera noise and JPEG compression blocks)
# @markdown - **80+**: Strong smoothing (good for heavily pixelated or noisy scans)
denoise_intensity = 50 # @param {"type":"slider","min":10,"max":100,"step":5}

# @markdown ---
# @markdown ### [4] Lighting, Shadows & Contrast
# @markdown **enable_contrast** (default: True): Balances shadows and highlights locally (CLAHE HDR effect).
enable_contrast = True # @param {type:"boolean"}

# @markdown **contrast_boost** (default: 2.0): How strongly to enhance shadows and highlights.
# @markdown - **1.0 - 1.5**: Subtle shadow lift
# @markdown - **2.0**: Balanced HDR clarity (recommended)
# @markdown - **3.0+**: High-contrast dramatic punch
contrast_boost = 2.0 # @param {"type":"slider","min":1.0,"max":4.0,"step":0.2}

# @markdown **brightness_shift** (default: 0): Overall exposure compensation.
# @markdown - **Negative (-10 to -30)**: Dims overly bright or washed-out photos
# @markdown - **0**: Original exposure untouched
# @markdown - **Positive (+10 to +30)**: Brightens dark, underexposed shots
brightness_shift = 0 # @param {"type":"slider","min":-50,"max":50,"step":5}

# @markdown ---
# @markdown ### [5] Color Vibrance & Temperature (White Balance)
# @markdown **vibrance_boost** (default: 1.1): Makes flat or faded colors pop.
# @markdown - **1.0**: Original saturation
# @markdown - **1.05 - 1.15**: Natural color boost (recommended)
# @markdown - **1.25+**: Vivid posterized colors
vibrance_boost = 1.1 # @param {"type":"slider","min":1.0,"max":1.5,"step":0.05}

# @markdown **color_temperature** (default: 0): Adjusts warmth and tint.
# @markdown - **Negative (-15 to -5)**: Cooler blue tone (fixes yellow indoor lighting)
# @markdown - **0**: Neutral / original tint
# @markdown - **Positive (+5 to +15)**: Warmer golden tone (sunlight warmth for portraits)
color_temperature = 0 # @param {"type":"slider","min":-30,"max":30,"step":5}

# @markdown ---
# @markdown ### [6] Output Format (default: PNG)
output_format = "PNG" # @param ["PNG", "JPEG", "WebP"]
# @markdown ---

import cv2, numpy as np, os
from PIL import Image, ImageOps
from google.colab import files
from IPython.display import display, HTML

print("Please select an image file to upload:")
uploaded = files.upload()

if not uploaded:
    print("No file selected!")
else:
    filename = list(uploaded.keys())[0]
    
    # 1. Load image and preserve correct orientation
    pil_in = Image.open(filename)
    pil_in = ImageOps.exif_transpose(pil_in)
    img_np = np.array(pil_in)
    
    # Handle RGB / RGBA / Grayscale channels
    alpha = None
    if img_np.ndim == 2:
        img = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
    elif img_np.shape[2] == 4:
        alpha = img_np[:, :, 3]
        img = cv2.cvtColor(img_np[:, :, :3], cv2.COLOR_RGB2BGR)
    else:
        img = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        
    h, w = img.shape[:2]
    
    # 2. Edge-Preserving Denoising (Bilateral Filter)
    if enable_denoise and denoise_intensity > 0:
        img = cv2.bilateralFilter(img, d=7, sigmaColor=float(denoise_intensity), sigmaSpace=float(denoise_intensity))
        
    # 3. High-Quality Lanczos-4 Upscaling
    if upscale_factor != 1:
        target_w = int(round(w * upscale_factor))
        target_h = int(round(h * upscale_factor))
        img = cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)
        if alpha is not None:
            alpha = cv2.resize(alpha, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)
            
    # 4. Adaptive Local Contrast (CLAHE on L-channel in LAB space)
    if enable_contrast and contrast_boost > 0:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=float(contrast_boost), tileGridSize=(8, 8))
        l = clahe.apply(l)
        img = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
        
    # 5. Brightness Adjustment
    if brightness_shift != 0:
        img = np.clip(img.astype(np.int16) + brightness_shift, 0, 255).astype(np.uint8)
        
    # 6. Unsharp Masking (Frequency sharpening)
    if sharpen_strength > 0:
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=float(sharpen_radius))
        unsharp = cv2.addWeighted(img, 1.0 + sharpen_strength, blur, -sharpen_strength, 0)
        img = np.clip(unsharp, 0, 255).astype(np.uint8)
        
    # 7. Color Vibrance Boost
    if vibrance_boost != 1.0:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * vibrance_boost, 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        
    # 8. Color Temperature (Cooler Blue vs Warmer Golden)
    if color_temperature != 0:
        b_ch, g_ch, r_ch = cv2.split(img.astype(np.float32))
        if color_temperature > 0:
            r_ch = np.clip(r_ch + color_temperature, 0, 255)
            b_ch = np.clip(b_ch - color_temperature * 0.5, 0, 255)
        else:
            b_ch = np.clip(b_ch - color_temperature, 0, 255)
            r_ch = np.clip(r_ch + color_temperature * 0.5, 0, 255)
        img = cv2.merge((b_ch, g_ch, r_ch)).astype(np.uint8)
        
    # Reassemble RGBA if alpha channel was present
    if alpha is not None and output_format == "PNG":
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        final_pil = Image.fromarray(np.dstack((rgb, alpha)))
    else:
        final_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
    # Save, preview, and trigger automatic download
    ext = output_format.lower()
    out_name = f"{os.path.splitext(filename)[0]}_enhanced_{upscale_factor}x.{ext}"
    
    if output_format == "JPEG":
        final_pil.convert("RGB").save(out_name, format="JPEG", quality=95)
    elif output_format == "WebP":
        final_pil.save(out_name, format="WEBP", quality=95)
    else:
        final_pil.save(out_name, format="PNG", compress_level=3)
        
    print(f"\nEnhanced successfully in ~0.05s!")
    print(f"Original: {w}x{h} -> Enhanced: {final_pil.width}x{final_pil.height} ({upscale_factor}x)")
    print(f"Saved as: {out_name}")
    # Responsive preview scaled to fit screen comfortably
    import base64
    from io import BytesIO
    preview = final_pil.copy()
    preview.thumbnail((1200, 800))
    buf = BytesIO()
    if preview.mode in ("RGBA", "LA"):
        preview.save(buf, format="PNG")
        mime = "image/png"
    else:
        preview.convert("RGB").save(buf, format="JPEG", quality=88)
        mime = "image/jpeg"
    b64_data = base64.b64encode(buf.getvalue()).decode()
    display(HTML(f'''
    <div style="margin: 14px 0;">
        <p style="margin-bottom: 8px; font-weight: 600; font-size: 14px;">Enhanced Preview ({final_pil.width}x{final_pil.height}):</p>
        <img src="data:{mime};base64,{b64_data}" style="max-width: 100%; max-height: 520px; border-radius: 8px; box-shadow: 0 4px 14px rgba(0,0,0,0.18); object-fit: contain;" />
    </div>
    '''))
    files.download(out_name)


In [ ]:
# @title Clear Storage
# @markdown Run this cell to delete uploaded and processed images from disk and free memory.

import os, glob, gc

deleted = 0
for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp"):
    for f in glob.glob(ext):
        try:
            os.remove(f)
            deleted += 1
        except OSError:
            pass

gc.collect()
print(f"Removed {deleted} temporary file(s).")
